In [2]:
import pandas as pd
df=pd.read_csv("Online Retail.csv")

In [3]:
# df.shape
# print(df.head(),"\n")
# print(df.columns,"\n")
# print(df.dtypes)

In [4]:
# print(df.isnull().sum(),"\n")
# print(df.duplicated().sum())

In [5]:
df=df.drop_duplicates()
df.shape
print(df.duplicated().sum())

print(df.isnull().sum())
df=df.dropna(subset=["CustomerID"])
print(df.isnull().sum())
df=df[df["Quantity"]>0]

df=df[df["UnitPrice"]>0]
df[df["UnitPrice"]<=0].shape
df.shape

0
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135037
Country             0
dtype: int64
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


(392692, 8)

In [6]:
# print("Current data : ",df.shape)
# print("Duplicate Values : ",df.duplicated().sum())
# print("Null Values : ",df["CustomerID"].isnull().sum())
# print("Negative Quantity : ",(df["Quantity"]<=0).sum())
# print("Negative price : ",(df["UnitPrice"]<=0).sum())

In [7]:
df["InvoiceDate"]=pd.to_datetime(df["InvoiceDate"])
df["CustomerID"]=df["CustomerID"].astype(int)
df.dtypes

InvoiceNo                 str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID              int64
Country                   str
dtype: object

In [8]:
df["TotalAmount"]=df["Quantity"]*df["UnitPrice"]
df.head

<bound method NDFrame.head of        InvoiceNo StockCode                          Description  Quantity  \
0         536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1         536365     71053                  WHITE METAL LANTERN         6   
2         536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3         536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4         536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   
...          ...       ...                                  ...       ...   
541904    581587     22613          PACK OF 20 SPACEBOY NAPKINS        12   
541905    581587     22899         CHILDREN'S APRON DOLLY GIRL          6   
541906    581587     23254        CHILDRENS CUTLERY DOLLY GIRL          4   
541907    581587     23255      CHILDRENS CUTLERY CIRCUS PARADE         4   
541908    581587     22138        BAKING SET 9 PIECE RETROSPOT          3   

               InvoiceDate  UnitPrice  Custom

In [9]:
df["CustomerID"].unique()
df.to_csv("cleaned_data.csv",index=False)

In [10]:
customer_df=df.groupby("CustomerID").size().reset_index(name="purchase_frequency")
customer_df["Totalspending"]=customer_df["CustomerID"].map(df.groupby("CustomerID")["TotalAmount"].sum())
customer_df["Last_perchased"]=customer_df["CustomerID"].map(df.groupby("CustomerID")["InvoiceDate"].max())
customer_df["Recency"]=(df["InvoiceDate"].max()-customer_df["Last_perchased"]).dt.days
customer_df["Churn"]=(customer_df["Recency"]>=90).astype(int)


In [11]:
customer_df.head()
end_date=df["InvoiceDate"].max()
print(end_date)
#customer_df["Churn"].value_counts()
#customer_df["Churn"].value_counts(normalize=True)*100
#customer_df.shape

2011-12-09 12:50:00


In [12]:
refrence_date=pd.Timestamp("2011-9-10")
print(refrence_date)
df_before=df[df["InvoiceDate"]<=refrence_date]
df_after=df[df["InvoiceDate"]>refrence_date]
print(df_after.shape)
print(df_before.shape)

2011-09-10 00:00:00
(158852, 9)
(233840, 9)


In [13]:
training_df=df_before.groupby("CustomerID").agg(
    purcahse_frequency=("InvoiceDate","nunique"),
    Total_spending=("TotalAmount","sum"),
    last_purchased=("InvoiceDate","max")
).reset_index()
training_df["recency"]=(refrence_date-training_df["last_purchased"]).dt.days

In [14]:
future_purchase=df_after[
    df_after["InvoiceDate"]<=pd.Timestamp("2011-12-9")].groupby("CustomerID").size()

In [15]:
training_df.head()
#future_purchase.head()
#training_df["churn"].value_counts()

,CustomerID,purcahse_frequency,Total_spending,last_purchased,recency
0,12346,1,77183.60,2011-01-18 10:01:00,234
1,12347,5,2790.86,2011-08-02 08:48:00,38
2,12348,3,1487.24,2011-04-05 10:47:00,157
3,12350,1,334.40,2011-02-02 16:01:00,219
4,12352,5,1561.81,2011-03-22 16:08:00,171


In [16]:
next_purcahse=df_after.groupby("CustomerID")["InvoiceDate"].min()
training_df["next_purchase"]=training_df["CustomerID"].map(next_purcahse)
training_df["Gap"]=(
    training_df["next_purchase"]-training_df["last_purchased"]
).dt.days


In [17]:
training_df["churn"]=(
    training_df["Gap"].isna()|(training_df["Gap"]>90)
).astype(int)

In [18]:
training_df["churn"].value_counts(normalize=True)*100
training_df.to_csv("Training_data.csv",index=False)

In [19]:
X=training_df[[
    "purcahse_frequency",
    "Total_spending",
    "recency"
]]
y=training_df["churn"]

In [20]:
X.head()
y.head()

0    1
1    0
2    1
3    1
4    1
Name: churn, dtype: int64

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [22]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(2696, 3)
(674, 3)
(2696,)
(674,)


In [23]:
# trainig ML model
model=LogisticRegression(max_iter=1000)
model.fit(X_train,y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [24]:
y_pred=model.predict(X_test)
print(y_pred[:20])
accuracy=accuracy_score(y_test,y_pred)
print("Accuracy : ",accuracy)


[0 0 1 1 0 1 0 1 0 1 0 1 1 0 0 1 1 1 1 1]
Accuracy :  0.8531157270029673
